# Sugestão de Compras

## Objetivo

Este notebook gera recomendações quantitativas de reposição para os produtos da **Distribuidora Horizonte**.

A sugestão considera:

- classificação ABC;
- venda média diária;
- estoque atual;
- estoque mínimo e máximo;
- dias de cobertura;
- prazo médio de entrega do fornecedor;
- estoque de segurança;
- compras realizadas e ainda não recebidas.

## Lógica

A necessidade de compra não é definida apenas pela quantidade atual em estoque.

São considerados:

1. consumo médio diário;
2. importância comercial do produto;
3. prazo médio do fornecedor;
4. estoque de segurança;
5. estoque já disponível;
6. quantidade já comprada e ainda não recebida.

## Política de estoque de segurança

- Classe A: 7 dias;
- Classe B: 5 dias;
- Classe C: 3 dias;
- Produtos sem venda: 0 dias.

Produtos Classe A recebem maior proteção devido à sua importância para o faturamento.

## Observação

A recomendação é uma simulação analítica para fins educacionais e de portfólio.

Em um ambiente real, a política de compras também poderia considerar lote mínimo, múltiplos de embalagem, descontos por volume, capacidade de armazenagem, validade dos produtos e previsão probabilística da demanda.

## Saídas

Este notebook cria:

- `sugestao_compras`;
- `resumo_sugestao_compras`.

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Silver: {schema_silver}")
print(f"Gold: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def carregar_gold(nome_tabela):
    """
    Carrega uma tabela da camada Gold.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):
    """
    Salva um DataFrame como tabela Delta
    na camada Gold.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
df_estoque = carregar_gold(
    "analise_estoque_atual"
)

df_produtos = carregar_silver(
    "dim_produto"
)

df_fornecedores = carregar_silver(
    "dim_fornecedor"
)

df_compras = carregar_silver(
    "fato_compras"
)

In [0]:
data_referencia = (

    df_estoque

    .agg(
        F.max(
            "data_referencia"
        ).alias(
            "data_referencia"
        )
    )

    .first()[
        "data_referencia"
    ]
)


print(
    f"Data de referência: "
    f"{data_referencia}"
)

In [0]:
df_compras_pendentes = (

    df_compras

    .filter(
        F.col(
            "data_recebimento"
        ).isNull()
    )

    .groupBy(
        "id_produto"
    )

    .agg(

        F.sum(
            "quantidade"
        ).alias(
            "quantidade_pendente_recebimento"
        ),

        F.round(
            F.sum(
                "valor_total"
            ),
            2
        ).alias(
            "valor_compras_pendentes"
        )
    )
)

In [0]:
display(
    df_compras_pendentes
    .orderBy(
        F.desc(
            "quantidade_pendente_recebimento"
        )
    )
    .limit(20)
)

In [0]:
df_dados_fornecimento = (

    df_produtos.alias("p")

    .join(
        df_fornecedores.alias("f"),

        F.col(
            "p.id_fornecedor_principal"
        )
        ==
        F.col(
            "f.id_fornecedor"
        ),

        how="left"
    )

    .select(

        F.col(
            "p.id_produto"
        ).alias(
            "id_produto"
        ),

        F.col(
            "p.id_fornecedor_principal"
        ).alias(
            "id_fornecedor"
        ),

        F.col(
            "f.nome_fornecedor"
        ).alias(
            "nome_fornecedor"
        ),

        F.col(
            "f.prazo_medio_entrega"
        ).alias(
            "prazo_medio_entrega"
        ),

        F.col(
            "f.avaliacao_fornecedor"
        ).alias(
            "avaliacao_fornecedor"
        ),

        F.col(
            "p.custo_unitario"
        ).alias(
            "custo_unitario"
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_estoque.alias("e")

    .join(
        df_dados_fornecimento.alias("f"),
        on="id_produto",
        how="left"
    )

    .join(
        df_compras_pendentes.alias("cp"),
        on="id_produto",
        how="left"
    )

    .fillna(
        {
            "quantidade_pendente_recebimento": 0,
            "valor_compras_pendentes": 0
        }
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "dias_estoque_seguranca",

        F.when(
            F.col("classe_abc") == "A",
            7
        )

        .when(
            F.col("classe_abc") == "B",
            5
        )

        .when(
            F.col("classe_abc") == "C",
            3
        )

        .otherwise(
            0
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "estoque_seguranca",

        F.ceil(
            F.col(
                "venda_media_diaria"
            )
            *
            F.col(
                "dias_estoque_seguranca"
            )
        ).cast("int")
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "demanda_durante_prazo",

        F.ceil(

            F.col(
                "venda_media_diaria"
            )

            *

            F.col(
                "prazo_medio_entrega"
            )

        ).cast("int")
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "ponto_reposicao",

        F.col(
            "demanda_durante_prazo"
        )
        +
        F.col(
            "estoque_seguranca"
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "estoque_projetado",

        F.col(
            "quantidade_estoque"
        )
        +
        F.col(
            "quantidade_pendente_recebimento"
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "dias_cobertura_alvo",

        F.when(
            F.col("classe_abc") == "A",
            30
        )

        .when(
            F.col("classe_abc") == "B",
            21
        )

        .when(
            F.col("classe_abc") == "C",
            14
        )

        .otherwise(
            0
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "estoque_alvo_demanda",

        F.ceil(
            F.col(
                "venda_media_diaria"
            )
            *
            F.col(
                "dias_cobertura_alvo"
            )
        ).cast("int")
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "estoque_alvo",

        F.when(
            F.col("classe_abc") == "Sem venda",
            0
        )

        .otherwise(

            F.least(

                F.col(
                    "estoque_maximo"
                ),

                F.greatest(

                    F.col(
                        "estoque_minimo"
                    ),

                    F.col(
                        "estoque_alvo_demanda"
                    )
                )
            )
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "necessita_reposicao",

        F.when(
            (
                F.col(
                    "estoque_projetado"
                )
                <=
                F.col(
                    "ponto_reposicao"
                )
            )
            &
            (
                F.col(
                    "venda_media_diaria"
                ) > 0
            ),

            1
        )

        .otherwise(0)
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "quantidade_sugerida_compra",

        F.when(
            F.col(
                "necessita_reposicao"
            ) == 1,

            F.greatest(

                F.col(
                    "estoque_alvo"
                )
                -
                F.col(
                    "estoque_projetado"
                ),

                F.lit(0)
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "valor_estimado_compra",

        F.round(
            F.col(
                "quantidade_sugerida_compra"
            )
            *
            F.col(
                "custo_unitario"
            ),
            2
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "prioridade_compra",

        F.when(
            (
                F.col(
                    "classe_abc"
                ) == "A"
            )
            &
            (
                F.col(
                    "quantidade_estoque"
                ) == 0
            ),

            "Crítica"
        )

        .when(
            (
                F.col(
                    "classe_abc"
                ) == "A"
            )
            &
            (
                F.col(
                    "necessita_reposicao"
                ) == 1
            ),

            "Muito alta"
        )

        .when(
            (
                F.col(
                    "classe_abc"
                ) == "B"
            )
            &
            (
                F.col(
                    "necessita_reposicao"
                ) == 1
            ),

            "Alta"
        )

        .when(
            (
                F.col(
                    "classe_abc"
                ) == "C"
            )
            &
            (
                F.col(
                    "necessita_reposicao"
                ) == 1
            ),

            "Normal"
        )

        .otherwise(
            "Sem necessidade"
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(

        "recomendacao",

        F.when(
            F.col(
                "prioridade_compra"
            ) == "Crítica",

            "Comprar imediatamente"
        )

        .when(
            F.col(
                "prioridade_compra"
            ) == "Muito alta",

            "Priorizar no próximo pedido"
        )

        .when(
            F.col(
                "prioridade_compra"
            ) == "Alta",

            "Programar reposição"
        )

        .when(
            F.col(
                "prioridade_compra"
            ) == "Normal",

            "Incluir no planejamento de compras"
        )

        .otherwise(
            "Não realizar nova compra neste momento"
        )
    )
)

In [0]:
df_sugestao_compras = (

    df_sugestao_compras

    .withColumn(
        "data_referencia",
        F.lit(
            data_referencia
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )

    .select(

        "data_referencia",

        "id_produto",
        "nome_produto",
        "categoria",

        "classe_abc",
        "ranking_faturamento",

        "faturamento_12m",
        "venda_media_diaria",

        "quantidade_estoque",
        "estoque_minimo",
        "estoque_maximo",

        "dias_cobertura",

        "dias_estoque_seguranca",
        "estoque_seguranca",

        "prazo_medio_entrega",
        "demanda_durante_prazo",

        "ponto_reposicao",

        "quantidade_pendente_recebimento",

        "estoque_projetado",

        "dias_cobertura_alvo",
        "estoque_alvo",

        "necessita_reposicao",

        "quantidade_sugerida_compra",

        "id_fornecedor",
        "nome_fornecedor",
        "avaliacao_fornecedor",

        "custo_unitario",
        "valor_estimado_compra",

        "prioridade_compra",
        "recomendacao",

        "_data_processamento"
    )
)

In [0]:
salvar_gold(
    df_sugestao_compras,
    "sugestao_compras"
)

In [0]:
display(

    df_sugestao_compras

    .filter(
        F.col(
            "quantidade_sugerida_compra"
        ) > 0
    )

    .select(

        "prioridade_compra",
        "classe_abc",

        "nome_produto",
        "categoria",

        "venda_media_diaria",

        "quantidade_estoque",
        "quantidade_pendente_recebimento",

        "ponto_reposicao",
        "estoque_alvo",

        "quantidade_sugerida_compra",

        "nome_fornecedor",

        "valor_estimado_compra"
    )

    .orderBy(

        F.when(
            F.col(
                "prioridade_compra"
            ) == "Crítica",
            1
        )
        .when(
            F.col(
                "prioridade_compra"
            ) == "Muito alta",
            2
        )
        .when(
            F.col(
                "prioridade_compra"
            ) == "Alta",
            3
        )
        .otherwise(4),

        F.desc(
            "valor_estimado_compra"
        )
    )
)

In [0]:
display(

    df_sugestao_compras

    .filter(
        (F.col("classe_abc") == "A")
        &
        (F.col("necessita_reposicao") == 1)
    )

    .select(

        "ranking_faturamento",
        "nome_produto",

        "quantidade_estoque",
        "dias_cobertura",

        "ponto_reposicao",
        "estoque_alvo",

        "quantidade_pendente_recebimento",
        "quantidade_sugerida_compra",

        "nome_fornecedor",
        "prazo_medio_entrega",

        "valor_estimado_compra",
        "prioridade_compra"
    )

    .orderBy(
        "ranking_faturamento"
    )
)

In [0]:
display(

    df_sugestao_compras

    .filter(
        F.col(
            "quantidade_pendente_recebimento"
        ) > 0
    )

    .select(
        "nome_produto",
        "classe_abc",
        "quantidade_estoque",
        "quantidade_pendente_recebimento",
        "estoque_projetado",
        "ponto_reposicao",
        "quantidade_sugerida_compra"
    )

    .orderBy(
        F.desc(
            "quantidade_pendente_recebimento"
        )
    )
)

In [0]:
df_resumo_sugestao_compras = (

    df_sugestao_compras

    .groupBy(
        "prioridade_compra"
    )

    .agg(

        F.count("*").alias(
            "quantidade_produtos"
        ),

        F.sum(
            "quantidade_sugerida_compra"
        ).alias(
            "quantidade_total_sugerida"
        ),

        F.round(
            F.sum(
                "valor_estimado_compra"
            ),
            2
        ).alias(
            "valor_estimado_compra"
        )
    )

    .withColumn(
        "data_referencia",
        F.lit(
            data_referencia
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_resumo_sugestao_compras,
    "resumo_sugestao_compras"
)

In [0]:
df_kpis_compras = (

    df_sugestao_compras

    .agg(

        F.sum(
            "necessita_reposicao"
        ).alias(
            "produtos_para_repor"
        ),

        F.sum(
            "quantidade_sugerida_compra"
        ).alias(
            "unidades_sugeridas"
        ),

        F.round(
            F.sum(
                "valor_estimado_compra"
            ),
            2
        ).alias(
            "investimento_estimado"
        ),

        F.sum(
            F.when(
                F.col(
                    "prioridade_compra"
                ) == "Crítica",
                1
            ).otherwise(0)
        ).alias(
            "compras_criticas"
        )
    )
)


display(
    df_kpis_compras
)

In [0]:
recomendacoes_invalidas = (

    df_sugestao_compras

    .filter(
        F.col(
            "quantidade_sugerida_compra"
        ) < 0
    )

    .count()
)


if recomendacoes_invalidas > 0:

    raise Exception(
        "Foram encontradas sugestões "
        "de compra negativas."
    )


print(
    "Validação concluída: "
    "nenhuma sugestão negativa."
)

In [0]:
sem_venda_com_compra = (

    df_sugestao_compras

    .filter(
        (F.col("classe_abc") == "Sem venda")
        &
        (
            F.col(
                "quantidade_sugerida_compra"
            ) > 0
        )
    )

    .count()
)


if sem_venda_com_compra > 0:

    raise Exception(
        "Produto sem venda gerou "
        "sugestão indevida de compra."
    )


print(
    "Validação concluída: "
    "produtos sem demanda não recebem reposição."
)

In [0]:
spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`sugestao_compras`
    IS 'Recomendação quantitativa de reposição considerando demanda, Curva ABC, estoque, prazo do fornecedor e compras pendentes.'
    """
)


spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`resumo_sugestao_compras`
    IS 'Resumo executivo das recomendações de compra por nível de prioridade.'
    """
)